# Figure 1, Panel E
Heatmap of ATAC-seq signal, log2 accessibility fold change (BAFinh/Vehicle), and ChIP-seq signal for promoters, enhancers, and insulator elements, annotated based on Ernst et al. 2011 (https://www.nature.com/articles/nature09906) ChromHMM 15-state model in GM12878. 

This notebook annotates consensus, filtered ATAC-seq peaks based on these annotations, generating the following outputs:
- all ATAC-seq peaks annotated with ChromHMM states (consensus_filt_annot_chromHMM.tsv)
- genomic coordinates (in BED format) for enhancer, insulator, and promoter elements

The latter are used in panel_e_hm.sh to generate a heatmap using the deepTools computeMatrix and plotHeatmap utilities. panel_e_hm.sh also uses the deepTools bigwigCompare tool to generate log2 accessibility fold change signal. 

Final outputs are:
- heatmap_matrix.gz -  *matrix from computeMatrix, used as input for heatmap generation*
- panel_e_heatmap.pdf - *final heatmap*

# Setup

In [1]:
#load packages
library(GenomicRanges)
library(rtracklayer)
library(tidyverse)
library(patchwork)
library(here)
here::i_am("figures/Figure1/PanelE/PanelE_chromHMM_annot.ipynb")
here()

sessionInfo()

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: GenomeInfoDb

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2

[1] "/mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics"

R version 4.2.2 Patched (2022-11-10 r83330)
Platform: x86_64-pc-linux-gnu (64-bit)
Running under: Ubuntu 23.04

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/libopenblasp-r0.3.21.so

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] here_1.0.2           patchwork_1.3.0      lubridate_1.9.4     
 [4] forcats_1.0.0        stringr_1.5.1        dplyr_1.1.4         
 [7] purrr_1.0.2          readr_2.1.5          tidyr_1.3.1         
[10] tibble_3.2.1         ggplot2_3.5.2        tidyverse_2.0.0     
[13] rtr

# Load inputs and annotate cREs

In [2]:
#Assign chromHMM annotations to ATAC-seq peaks

#read in BED files:
# ATAC peaks
atac <- import(
  here(
    "inputs", "bed", "ATAC_peaks", "filtered",
    "consensus_filt.bed"
  )
)

# chromHMM annotations (lifted over from hg19)
chromhmm <- import(
  here(
    "inputs", "cRE", "chromHMM",
    "hg38_wgEncodeBroadHmmGm12878HMM.bed"
  )
)

# row metadata
atac_meta <- read.delim(
  here(
    "inputs", "bed", "ATAC_peaks", "filtered",
    "consensus_filt_annot.tsv"
  ),
  sep = "\t"
)

# find overlaps between ATAC and chromHMM regions
hits <- findOverlaps(atac, chromhmm)

# 3. Make a data frame of results
overlap_df <- data.frame(
    peak_id = queryHits(hits), #row # in ATAC peaks data frame
    chromHMM_id = subjectHits(hits), #row # in chromHMM data frame
    peak_chr = seqnames(atac)[queryHits(hits)], #ATAC-seq peak coordinates
    peak_start = start(atac)[queryHits(hits)],
    peak_end = end(atac)[queryHits(hits)],
    chromHMM_chr = seqnames(chromhmm)[subjectHits(hits)], #chromHMM state coordinates
    chromHMM_start = start(chromhmm)[subjectHits(hits)],
    chromHMM_end = end(chromhmm)[subjectHits(hits)],
    chromHMM_state  = mcols(chromhmm)$name[subjectHits(hits)]  #chromHMM state
)

#multiple chromatin states can overlap one ATAC-seq peak. Here, applying the rule to keep only the state with the largest overlap
overlap_df <- overlap_df %>%
  mutate(overlap_bp = pmin(peak_end, chromHMM_end) - pmax(peak_start, chromHMM_start) + 1) %>% #compute overlaps
  group_by(peak_id) %>%
  slice_max(overlap_bp, n = 1) %>%
  ungroup()

#add chromHMM state annotations to ATAC-seq peak data frame
atac$chromHMM_state <- NA
atac$chromHMM_state[overlap_df$peak_id] <- overlap_df$chromHMM_state

#add TSS overlap to atac GR
atac$TSS <- atac_meta$TSS

Warning message in .merge_two_Seqinfo_objects(x, y):
“Each of the 2 combined objects has sequence levels not in the other:
  - in 'x': chrY
  - in 'y': chr4_GL000008v2_random, chrUn_KI270742v1, chr1_KI270706v1_random, chr14_GL000009v2_random, chr2_KI270894v1_alt, chr7_KI270803v1_alt, chr15_KI270850v1_alt, chr17_KI270909v1_alt, chr22_KI270879v1_alt
  Make sure to always combine/compare objects based on the same reference
  genome (use suppressWarnings() to suppress this warning).”


# Simplify cRE annotations
Enhancer and promoter states combined into single annotations (see Supplemental Figure 1F)

In [3]:
#summarize chromHMM_states by collapsing like states ('Enhancer', 'Promoter', 'Txn') into single annotations

atac_df <- atac %>% data.frame() %>% 
                mutate(chromHMM_state_slim = case_when(
                    grepl("Promoter",chromHMM_state) ~ "Promoter",
                    grepl("Enhancer", chromHMM_state) ~ "Enhancer",
                    grepl("Txn", chromHMM_state) ~ "Transcribed",
                    chromHMM_state == "8_Insulator" ~ "Insulator",
                    chromHMM_state == "12_Repressed" ~ "PC_repressed", 
                    TRUE ~ "Other"),
                       cRE_type = atac_meta$type) #also add ChIP peak-overlap cRE annotations

# Write outputs

In [4]:
#write out table
write.table(atac_df, "consensus_filt_annot_chromHMM.tsv", sep='\t',row.names=FALSE)

#write out BED files for different states
write.table(atac_df[atac_df$chromHMM_state_slim == "Enhancer",1:3], "chromHMM_Enhancer.bed", col.names=FALSE, sep='\t', row.names=FALSE, quote=FALSE)
write.table(atac_df[atac_df$chromHMM_state_slim == "Promoter",1:3], "chromHMM_Promoter.bed", col.names=FALSE, sep='\t', row.names=FALSE, quote=FALSE)
write.table(atac_df[atac_df$chromHMM_state_slim == "Insulator",1:3], "chromHMM_Insulator.bed", col.names=FALSE, sep='\t', row.names=FALSE, quote=FALSE)